# 迭代器、生成器、属性与正则表达式

本篇对应《Python 进阶》第 5 天《Python 高级与正则表达式》讲义，按课程顺序整理四部分：迭代器、生成器、`property` 属性和正则表达式。

课程的共同主线是“按规则访问或处理数据”：迭代器逐个访问集合元素，生成器按需产生数据，`property` 把方法封装成属性，正则表达式按模式匹配、检索和提取字符串。课程代码均保留在 Markdown 代码围栏中，不在本笔记中执行，也不伪造运行结果。

## 一、迭代器

### 1. 什么是迭代器

迭代器（Iterator）是一种用于逐个访问数据集合元素的对象，调用方不需要了解集合的底层实现。迭代器通过统一的方式遍历列表、元组等可迭代对象，适合按需读取数据。

一个完整的迭代器对象需要实现两个方法：

- `__iter__()`：返回迭代器对象本身；
- `__next__()`：返回下一个元素；没有更多元素时抛出 `StopIteration`。

迭代器通常只保存当前位置和结束条件，不必一次性把全部结果复制到另一份内存中，因此适合处理大型集合或流式数据。

> 讲义把 `range()` 作为迭代器示例。严格区分时，`range` 是可迭代对象，`iter(range(...))` 返回的对象才是迭代器；实际使用时两者都可以直接放进 `for` 循环。

### 2. 自定义迭代器

下面的类用 `current` 保存当前位置，用 `end` 保存结束条件。`__next__()` 在越过终点时抛出 `StopIteration`，这正是 `for` 循环判断遍历结束的信号。

```python
class MyIterator:
    def __init__(self, start, end):
        self.current = start
        self.end = end

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.end:
            raise StopIteration
        value = self.current
        self.current += 1
        return value


iterator = MyIterator(1, 5)
for num in iterator:
    print(num)
```

实现要点：

1. `__iter__()` 必须返回一个可继续调用 `__next__()` 的对象，通常就是 `self`。
2. 每次 `__next__()` 只返回一个值，并更新内部状态。
3. 结束条件必须明确抛出 `StopIteration`，否则 `for` 循环无法正常结束。

### 3. `next()` 与 `for` 遍历

可以手动调用 `next(iterator)` 获取一个元素；也可以交给 `for` 循环自动调用 `__iter__()` 和 `__next__()`。`for` 循环会自动处理 `StopIteration`，所以不需要手动捕获它。

```python
iterator = MyIterator(1, 3)

print(next(iterator))
print(next(iterator))

for value in iterator:
    print(value)
```

手动调用 `next()` 后，迭代器的位置已经向后移动，后续 `for` 循环会从当前位置继续，而不是重新开始。迭代器通常是一次性的；如果需要再次遍历，应重新创建迭代器。

### 4. 迭代器小结

| 项目 | 说明 |
| --- | --- |
| 定义 | 按顺序访问集合元素的对象，不暴露集合底层实现 |
| 协议 | 实现 `__iter__()` 和 `__next__()` |
| 结束 | 没有元素时抛出 `StopIteration` |
| 使用 | `next()` 手动取值，或用 `for` 自动遍历 |
| 优势 | 支持惰性计算，适合大数据和流数据 |

## 二、生成器

### 1. 什么是生成器

生成器是一种按照既定规则逐个生成数据的机制。它不会一次性生成全部结果，而是“需要一个，生成一个”，因此可以节省大量内存。

生成器本身也是迭代器，常见创建方式有两种：

1. 生成器表达式；
2. 在函数中使用 `yield` 关键字。

生成器的核心是惰性计算：创建生成器时通常只保存生成规则，真正的计算会推迟到调用 `next()` 或执行 `for` 遍历时。

### 2. 生成器表达式

把列表推导式的方括号改成圆括号，可以得到生成器表达式。圆括号表示“这是一个生成器”，括号中的内容是生成规则，不是已经存好的结果。

```python
my_generator = (i * 2 for i in range(5))

# 这里只能看到生成器对象，数据尚未全部生成
print(my_generator)

# 每次取出一个值
value = next(my_generator)
print(value)

# 继续从当前位置遍历
for value in my_generator:
    print(value)
```

注意：生成器被 `next()` 取走的元素不会再次出现在后面的 `for` 循环中；生成器耗尽后，再调用 `next()` 会抛出 `StopIteration`。

### 3. `yield` 生成器函数

只要函数体中出现 `yield`，调用这个函数时就不会立即执行完整函数体，而会返回一个生成器。执行到 `yield` 时，函数把后面的值交给调用者，并暂停在当前位置；下一次 `next()` 会从暂停处继续。

```python
def my_generator(n):
    for i in range(n):
        print("开始生成...")
        yield i
        print("完成一次...")


g = my_generator(2)

# 手动获取下一个值
result = next(g)
print(result)

# for 循环内部会自动调用 next()，并处理 StopIteration
for value in g:
    print(value)
```

`yield` 的两个关键特性：

- 把 `yield` 后面的值返回给调用方；
- 在 `yield` 所在位置暂停，保留局部变量和循环状态。

### 4. 生成器的应用：按批次加载数据

训练模型时，通常不会把全部样本一次性送入模型，而是按批次读取。下面的 `dataset_loader()` 读取歌词文件，计算批次数，再用 `yield` 每次返回一批数据。

```python
import math


def dataset_loader(batch_size):
    # 1. 读取歌词
    with open("./jaychou_lyrics.txt", "r", encoding="utf-8") as file:
        lines = file.readlines()

    # 2. 统计数据条数
    lyrics_number = len(lines)

    # 3. 向上取整，计算批次数
    batch_number = math.ceil(lyrics_number / batch_size)

    # 4. 逐批生成数据
    for idx in range(batch_number):
        start = idx * batch_size
        end = start + batch_size
        yield lines[start:end]


dataloader = dataset_loader(8)
for data in dataloader:
    print(data)
```

这种写法每次只保留当前批次，适合向模型逐批喂数据，也为后续 AI 专业课中的数据加载器打基础。实际运行前要确认歌词文件路径和编码正确。

### 5. 迭代器与生成器的区别

| 对比项 | 迭代器 | 生成器 |
| --- | --- | --- |
| 实现方式 | 手动实现 `__iter__()`、`__next__()` | 使用生成器表达式或 `yield` |
| 状态管理 | 自己保存当前位置和结束条件 | Python 自动保存暂停位置和局部状态 |
| 代码量 | 通常较多 | 更简洁 |
| 内存 | 通常可以惰性计算 | 天然按需生成，适合大数据或流数据 |
| 适用场景 | 需要高度控制迭代逻辑 | 需要简洁地生成序列或批次数据 |

> 生成器不是另一种完全不同的遍历协议；它是 Python 用来更简洁地创建迭代器的一种方式。

## 三、`property` 属性

### 1. `property` 的作用

`property` 可以把一个方法当作属性使用，从而在不改变外部调用方式的情况下，把读取或修改属性的逻辑集中到方法中。例如，外部使用 `person.age`，类内部仍可以在读取和赋值时进行校验、转换或其他处理。

定义 `property` 属性有两种课程写法：

- `@property` 装饰器方式；
- `property(getter, setter)` 类属性方式。

### 2. 装饰器方式

`@property` 修饰获取值的方法，`@age.setter` 修饰设置值的方法。外部读取和赋值时都使用属性语法，不需要显式调用方法。

```python
class Person(object):
    def __init__(self):
        self.__age = 0

    @property
    def age(self):
        return self.__age

    @age.setter
    def age(self, new_age):
        self.__age = new_age


p1 = Person()
print(p1.age)
p1.age = 100
print(p1.age)
```

如果只定义 `@property` 而没有定义 setter，属性可以读取但不能通过赋值修改。私有属性 `__age` 仍由类内部管理，外部只接触 `age` 这个接口。

### 3. 类属性方式

也可以先分别编写获取和设置方法，再使用 `property()` 把它们组合成一个属性。

```python
class Person(object):
    def __init__(self):
        self.__age = 0

    def get_age(self):
        return self.__age

    def set_age(self, new_age):
        self.__age = new_age

    age = property(get_age, set_age)


p1 = Person()
print(p1.age)
p1.age = 100
print(p1.age)
```

两种写法实现的目标相同：把“读取方法”和“设置方法”映射为属性访问。装饰器方式更紧凑，`property()` 方式更直接地展示了属性与方法的绑定关系。

### 4. `property` 易错点

- `@property` 后面的方法名决定属性名，例如方法名是 `age`，外部就使用 `p1.age`。
- setter 装饰器必须写成 `@age.setter`，其中 `age` 要与 property 名一致。
- setter 方法必须接收新值参数。
- 不要在 getter 中再次访问 `self.age`，否则会递归调用 getter；应访问内部属性，例如 `self.__age`。

## 四、正则表达式概述与 `re` 模块

### 1. 为什么学习正则表达式

实际开发中经常需要查找符合某种规则的字符串，例如邮箱、图片地址、手机号码或 IP 地址。正则表达式用一组特殊符号描述字符串匹配模式，可以完成：

- 数据验证：验证表单、手机、邮箱、IP 地址等；
- 数据检索：从文本中查找或抓取内容；
- 数据隐藏：例如把手机号显示为 `135****6235`；
- 数据过滤：过滤论坛中的敏感关键词。

正则表达式并非 Python 独有，Java、PHP、Go、JavaScript 等语言也支持类似的模式匹配。

### 2. `re` 模块的基本使用步骤

Python 中使用 `re` 模块完成正则匹配，基本流程是：

1. 导入 `re` 模块；
2. 使用 `match()`、`search()` 等方法匹配；
3. 匹配成功后，用 `group()` 提取结果。

```python
import re

result = re.match(pattern, string, flags=0)
if result:
    print(result.group())
```

`flags` 是可选参数，用于指定匹配模式，例如忽略大小写、多行匹配等。`match()` 从字符串开头匹配；如果需要扫描整段字符串并查找第一个符合项，使用 `search()`。

### 3. `match()`、`search()` 与替换

```python
import re

# match：必须从字符串开头开始匹配
result = re.match(r".it.", "aitcast")
if result:
    print(result.group())

# search：扫描字符串，返回第一个成功匹配
result = re.search(r"\d.*", "city:1beijing2.shanghai")
if result:
    print(result.group())

# compile：先编译模式，再重复使用
sentence = "车主说：你的刹车片应该更换了啊，嘿嘿"
pattern = re.compile(r"呢|吧|哈|啊|啦|嘿|嘿嘿")
clean_sentence = pattern.sub("", sentence)
print(clean_sentence)

# 字符串的 replace() 适合固定字符串替换
text = "你好."
print(text.replace(".", "。"))
```

正则替换适合“按规则删除或替换”；普通 `str.replace()` 适合精确替换固定文本。课程还演示了用 `[^...]` 保留汉字、数字、字母和指定标点，编写字符范围时要注意反斜杠和连字符的转义。

常用 API 的选择：

| 方法 | 用途 | 返回结果 |
| --- | --- | --- |
| `match()` | 从字符串开头匹配 | `Match` 对象或 `None` |
| `search()` | 扫描字符串，查找第一个匹配 | `Match` 对象或 `None` |
| `findall()` | 找出全部匹配内容 | 列表 |
| `sub()` | 按模式替换匹配内容 | 新字符串 |
| `split()` | 按模式切分字符串 | 列表 |

```python
import re

text = "编号：12，编号：34"
print(re.findall(r"\d+", text))
print(re.split(r"，", text))
```

`findall()` 适合一次提取全部结果；`split()` 的分隔符由正则模式决定。使用分组后，`findall()` 的返回结构会受到分组影响，编写模式时要提前确认希望得到完整匹配还是分组内容。

## 五、正则表达式编写

### 1. 匹配单个字符

| 写法 | 含义 |
| --- | --- |
| `.` | 匹配任意 1 个字符（默认不包含换行） |
| `[abc]` | 匹配方括号中列出的任意 1 个字符 |
| `[^abc]` | 匹配除方括号中字符以外的任意 1 个字符 |
| `\d` | 匹配数字，等价于 `[0-9]` |
| `\D` | 匹配非数字 |
| `\s` | 匹配空白字符，如空格、Tab |
| `\S` | 匹配非空白字符 |
| `\w` | 匹配字母、数字、下划线及汉字等非特殊字符 |
| `\W` | 匹配特殊字符，即非字母、非数字、非下划线、非汉字 |

这些符号默认一次匹配一个字符，实际匹配仍受 `match()` 从字符串开头匹配的规则约束。

### 2. 单字符匹配示例

```python
import re

result = re.match(r"itcast.", "itcast2")
if result:
    print(result.group())

result = re.match(r"itcast[123abc]", "itcast376")
if result:
    print(result.group())

result = re.match(r"itcast\d", "itcast5")
if result:
    print(result.group())

result = re.match(r"itcast\D", "itcast-")
if result:
    print(result.group())

result = re.match(r"itcast\s111", "itcast\t111")
if result:
    print(result.group())

result = re.match(r"itcast\w", "itcasta")
if result:
    print(result.group())
```

写正则时优先使用原始字符串 `r"..."`，这样可以减少 Python 字符串本身对反斜杠的二次转义。

### 3. 匹配多个字符：数量词

数量词修饰它前面的一个字符或一个分组：

| 写法 | 含义 |
| --- | --- |
| `*` | 前一个字符出现 0 次或无限次，可有可无 |
| `+` | 前一个字符出现 1 次或无限次，至少 1 次 |
| `?` | 前一个字符出现 0 次或 1 次 |
| `{m}` | 前一个字符出现恰好 `m` 次 |
| `{m,n}` | 前一个字符出现 `m` 到 `n` 次 |

```python
import re

# 数字可以出现 0 次或多次
result = re.match(r"itcast\d*itcast", "itcast123333itcast")

# 数字至少出现 1 次
result = re.match(r"itcast\d+itcast", "itcast11333444itcast")

# 数字出现 0 次或 1 次
result = re.match(r"itcast\d?itcast", "itcastitcast")

# 数字恰好出现 2 次
result = re.match(r"itcast\d{2}itcast", "itcast12itcast")

# 数字出现 2 到 5 次
result = re.match(r"itcast\d{2,5}itcast", "itcast111itcast")
```

`{m,n}` 中逗号后不要写空格，例如 `{2, 5}` 不是本课示例中的写法。

### 4. 匹配开头、结尾与排除字符

| 写法 | 含义 |
| --- | --- |
| `^` | 匹配字符串开头 |
| `$` | 匹配字符串结尾 |
| `[^...]` | 匹配不属于指定字符集合的字符 |

```python
import re

# 以数字开头
result = re.match(r"^\ditcast", "2itcast")

# 以数字开头，后面跟任意字符
result = re.match(r"^\d.*", "22itcast")

# 以数字结尾
result = re.match(r".*\d$", "itcast66")

# 同时要求以数字开头、以数字结尾
result = re.match(r"^\d.*\d$", "11itcast22")

# 以数字开头，结尾不能是 4
result = re.match(r"^\d.*[^4]$", "11itcast@")
```

`^` 和 `$` 表达的是位置，不会额外消耗一个字符；`[^4]` 表示最后一个字符不是 `4`，并不是“匹配一个数字”。

### 5. 分组、选择与分组引用

| 写法 | 含义 |
| --- | --- |
| `|` | 匹配左右任意一个表达式 |
| `(ab)` | 把括号中的内容作为一个分组 |
| `\num` | 引用编号为 `num` 的分组已匹配到的字符串 |

#### 选择与邮箱匹配

`|` 的优先级容易造成误解。邮箱的域名部分需要整体分组，否则表达式会被解释成多个不完整的分支。点号 `.` 在正则中代表任意字符，匹配真正的点时要写成 `\.`。

```python
import re

fruit = ["apple", "banana", "orange", "pear"]
for value in fruit:
    result = re.match(r"apple|pear", value)
    if result:
        print("我想吃的水果：", value)

email = "hello@qq.com"
result = re.match(
    r"[a-zA-Z0-9_]{4,20}@(163|126|qq)\.com",
    email,
)
if result:
    print(result.group())
```

#### `group()` 与分组编号

`group()` 或 `group(0)` 表示完整匹配结果；`group(1)`、`group(2)` 等表示从左到右编号的分组。

```python
import re

result = re.match(r"(qq):([1-9]\d{4,11})", "qq:10567")
if result:
    print(result.group(0))  # qq:10567
    print(result.group(1))  # qq
    print(result.group(2))  # 10567
```

#### 用分组引用匹配成对标签

如果开始标签和结束标签的名字必须相同，可以用 `\1` 引用第一个分组。使用原始字符串可避免 Python 字符串再次解释反斜杠。

```python
import re

result = re.match(r"<([a-zA-Z1-6]{4})>.*</\1>", "<html>hh</html>")
if result:
    print(result.group())
```

### 6. 正则表达式练习

按讲义要求，可以用前面的语法完成以下练习：

1. 在 `["apple", "banana", "orange", "pear"]` 中匹配 `apple` 和 `pear`。
2. 匹配 `163`、`126`、`qq` 等邮箱域名。
3. 匹配 `qq:10567`，分别提取 `qq` 文字和号码。
4. 匹配 `<html>hh</html>`，要求开始和结束标签名称一致。
5. 进一步匹配 `<html><h1>www.itcast.cn</h1></html>`。

完成练习时按以下顺序排查：先确认从哪里开始匹配，再确认字符类和数量词，最后检查分组、转义和开头/结尾锚点。

## 六、本篇要点回顾

- 迭代器通过 `__iter__()` 和 `__next__()` 按顺序提供数据，结束时抛出 `StopIteration`。
- 生成器可以用生成器表达式或 `yield` 创建，按需生成数据，适合大数据和批次加载。
- `property` 把方法封装成属性，装饰器方式和 `property()` 方式都可以实现读取与设置。
- 正则表达式用模式描述字符串规则；`match()` 从开头匹配，`search()` 扫描并查找，`group()` 提取结果，`sub()` 按规则替换。
- 写正则时重点检查字符类、数量词、`^`/`$`、分组、反斜杠转义，以及 `|` 的分支范围。